In [1]:
!pip install openai

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="APIKEY_HER")  # ← lim inn API-nøkkelen din her

import json
from pathlib import Path
import pandas as pd

def generate_gpt_variants(prompt, max_variants=5):
    prefix = "Translate the following to ASL:\n"
    if not prompt.startswith(prefix):
        print(f"Ugyldig prompt-format: {prompt}")
        return []

    original_sentence = prompt[len(prefix):].strip()

    system_msg = (
        "Du er en hjelpsom assistent som skriver om engelske setninger til nye varianter "
        "som betyr det samme, men med forskjellig ordvalg og struktur. "
        "Svar kun med variasjonene, én per linje. Ingen nummerering, ingen ekstra tekst."
    )
    user_msg = f"Lag {max_variants} varianter av denne setningen:\n\"{original_sentence}\""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            temperature=0.6,
        )
        lines = response.choices[0].message.content.splitlines()
        return [f"{prefix}{line.strip('•- ')}" for line in lines if line.strip()]
    except Exception as e:
        print(f"Feil for prompt: {prompt} → {e}")
        return []

# Last inn originalfil
original_path = "/content/asl_dataset.jsonl"  # Endre filsti om nødvendig
with open(original_path, "r", encoding="utf-8") as f:
    original_data = [json.loads(line) for line in f]

augmented = []
for row in original_data:
    prompt, completion = row["prompt"], row["completion"]
    augmented.append({"prompt": prompt, "completion": completion})

    variants = generate_gpt_variants(prompt, max_variants=5)
    for v in variants:
        augmented.append({"prompt": v, "completion": completion})

# Lagre utvidet datasett
output_path = "/content/asl_dataset_gpt_augmented.jsonl"
with open(output_path, "w", encoding="utf-8") as f:
    for row in augmented:
        json.dump(row, f, ensure_ascii=False)
        f.write("\n")

print(f"✅ Ferdig! Lagret som {output_path}")

KODEN UNDER ER TESTVARIANT FOR 1 Prompt med print her i Collab

In [ ]:
import json
from openai import OpenAI, RateLimitError

# Sett inn din OpenAI API-nøkkel her
client = OpenAI(api_key="APIKEY_HER")  # ← Bytt ut med din faktiske nøkkel

def generate_gpt_variants(prompt, max_variants=5):
    prefix = "Translate the following to ASL:\n"
    if not prompt.startswith(prefix):
        print(f"⛔️ Ugyldig prompt-format: {prompt}")
        return []

    original_sentence = prompt[len(prefix):].strip()

    system_msg = (
      "Du er en hjelpsom assistent som skriver om engelske setninger til nye varianter "
      "som betyr det samme og bruker lignende stil, tone og ordvalg. "
      "Hold deg nær originalens struktur og ordlyd, bare gjør små variasjoner. "
      "Lag X varianter av denne setningen som har samme mening og lignende stil"
      "(hverken for formelle eller for sleng), og som kunne brukes i samme situasjon."
      "Svar kun med variasjonene, én per linje. Ingen nummerering, ingen ekstra tekst."
      "Det er essensielt at setningen holder samme logiske betydning."
    )

    user_msg = f"Lag {max_variants} varianter av denne setningen:\n\"{original_sentence}\""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            temperature=0.2,
        )
        lines = response.choices[0].message.content.splitlines()
        return [f"{prefix}{line.strip('•- ')}" for line in lines if line.strip()]
    except RateLimitError:
        print("❌ Rate limited – prøv igjen senere.")
    except Exception as e:
        print(f"❌ Feil ved API-kall: {e}")
    return []

# Test: Last inn én prompt fra datasettet
test_path = "/content/asl_dataset.jsonl"
with open(test_path, "r", encoding="utf-8") as f:
    line = f.readline()
    test_row = json.loads(line)

prompt = test_row["prompt"]
completion = test_row["completion"]

print(f"\n📤 Prompt: {prompt}")
variants = generate_gpt_variants(prompt, max_variants=5)

print("\n📥 Genererte varianter:")
for i, v in enumerate(variants, 1):
    print(f"{i}. {v}")



📤 Prompt: Translate the following to ASL:
 I’ve never heard of that!

📥 Genererte varianter:
1. Translate the following to ASL:
I've never come across that before!
2. Translate the following to ASL:
That's news to me!
3. Translate the following to ASL:
I've never encountered that before!
4. Translate the following to ASL:
That's a new one for me!
5. Translate the following to ASL:
I've never been aware of that before!


KODEN UNDER ER TEST FOR 1 PROMPT SOM SKRIVER TIL NY FIL

In [ ]:
import json
from openai import OpenAI, RateLimitError

# Initialiser OpenAI-klienten
client = OpenAI(api_key="APIKEY_HER")

def generate_gpt_variants(prompt, max_variants=5):
    prefix_old = "Translate the following to ASL:\n"
    prefix_new = "Translate the following to American Sign Language:\n"

    if not prompt.startswith(prefix_old):
        print(f"⛔️ Ugyldig prompt-format: {prompt}")
        return []

    original_sentence = prompt[len(prefix_old):].strip()

    system_msg = (
        "Du er en hjelpsom assistent som skriver om engelske setninger til nye varianter "
        "som betyr det samme og bruker lignende stil, tone og ordvalg. "
        "Hold deg nær originalens struktur og ordlyd, bare gjør små variasjoner. "
        "Svar kun med variasjonene, én per linje. Ingen nummerering, ingen ekstra tekst. "
        "Det er essensielt at setningen beholder samme logiske betydning."
    )
    user_msg = f"Lag {max_variants} varianter av denne setningen:\n\"{original_sentence}\""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            temperature=0.2,
        )
        lines = response.choices[0].message.content.splitlines()
        return [f"{prefix_new}{line.strip('•- ')}" for line in lines if line.strip()]
    except RateLimitError:
        print("❌ Rate limited – prøv igjen senere.")
    except Exception as e:
        print(f"❌ Feil ved API-kall: {e}")
    return []

# Last inn én prompt+completion fra datasettet
test_path = "/content/asl_dataset.jsonl"
with open(test_path, "r", encoding="utf-8") as f:
    line = f.readline()
    test_row = json.loads(line)

original_prompt = test_row["prompt"]
completion = test_row["completion"]

print(f"\n📤 Original prompt:\n{original_prompt}")
print(f"🎯 Completion:\n{completion}")

# Generer varianter
variants = generate_gpt_variants(original_prompt, max_variants=5)

# Skriv til fil i JSONL-format
output_path = "/content/asl_dataset_variants_test.jsonl"
with open(output_path, "w", encoding="utf-8") as f:
    # Først originalen, men med oppdatert prompt-format
    orig_prompt_new = "Translate the following to American Sign Language:\n" + original_prompt.split(":",1)[-1].strip()
    f.write(json.dumps({"prompt": orig_prompt_new, "completion": completion}, ensure_ascii=False) + "\n")
    # Så variantene
    for v in variants:
        f.write(json.dumps({"prompt": v, "completion": completion}, ensure_ascii=False) + "\n")

print(f"\n✅ Lagret {1 + len(variants)} rader til {output_path}")



📤 Original prompt:
Translate the following to ASL:
 I’ve never heard of that!
🎯 Completion:
NEVER EAR THAT!

✅ Lagret 6 rader til /content/asl_dataset_variants_test.jsonl


ENDELIG KODE SOM 5x DATASETT

In [ ]:
import json
import time
from openai import OpenAI, RateLimitError

# Initialiser OpenAI-klienten med API-nøkkelen din
client = OpenAI(api_key="APIKEY_HER")

def generate_gpt_variants(prompt, max_variants=5):
    prefix_old = "Translate the following to ASL:\n"
    prefix_new = "Translate the following to American Sign Language:\n"

    if not prompt.startswith(prefix_old):
        print(f"⛔️ Ugyldig prompt-format: {prompt}")
        return []

    original_sentence = prompt[len(prefix_old):].strip()

    system_msg = (
        "Du er en hjelpsom assistent som skriver om engelske setninger til nye varianter "
        "som betyr det samme og bruker lignende stil, tone og ordvalg. "
        "Hold deg nær originalens struktur og ordlyd, bare gjør små variasjoner. "
        "Svar kun med variasjonene, én per linje. Ingen nummerering, ingen ekstra tekst. "
        "Det er essensielt at setningen beholder samme logiske betydning."
    )
    user_msg = f"Lag {max_variants} varianter av denne setningen:\n\"{original_sentence}\""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            temperature=0.2,
        )
        lines = response.choices[0].message.content.splitlines()
        return [f"{prefix_new}{line.strip('•- ')}" for line in lines if line.strip()]
    except RateLimitError:
        print("❌ Rate limited – prøver igjen om 10 sekunder...")
        time.sleep(10)
        return generate_gpt_variants(prompt, max_variants)
    except Exception as e:
        print(f"❌ Feil ved API-kall: {e}")
        return []

# Last inn hele datasettet
input_path = "/content/asl_dataset_kopi.jsonl"
with open(input_path, "r", encoding="utf-8") as f:
    original_data = [json.loads(line) for line in f]

# Skriv utvidet datasett til ny fil
output_path = "/content/asl_dataset_variants_full.jsonl"
with open(output_path, "w", encoding="utf-8") as f:
    total = 0
    for i, row in enumerate(original_data, 1):
        prompt_old = row["prompt"]
        completion = row["completion"]

        # Oppdatert prompt med nytt prefix
        prefix_new = "Translate the following to American Sign Language:\n"
        original_sentence = prompt_old.split(":", 1)[-1].strip()
        new_prompt = f"{prefix_new}{original_sentence}"

        # Skriv original
        f.write(json.dumps({"prompt": new_prompt, "completion": completion}, ensure_ascii=False) + "\n")
        total += 1

        # Generer varianter
        variants = generate_gpt_variants(prompt_old, max_variants=5)
        for v in variants:
            f.write(json.dumps({"prompt": v, "completion": completion}, ensure_ascii=False) + "\n")
            total += 1

        print(f"✅ {i}/{len(original_data)} rader prosessert – totalt {total} rader skrevet", end="\r")

print(f"\n🎉 Ferdig! Filen er lagret som: {output_path}")


✅ 504/504 rader prosessert – totalt 3024 rader skrevet
🎉 Ferdig! Filen er lagret som: /content/asl_dataset_variants_full.jsonl
